In [ ]:
import csv
import pandas as pd
from rank_bm25 import BM25Okapi
from tqdm import tqdm

BASE_PATH = r"C:\Users\nihal\OneDrive\Desktop\NLP_Project"

PASSAGES_PATH = BASE_PATH + r"\subset_passages.tsv"
QUERIES_PATH = BASE_PATH + r"\subset_queries.tsv"

OUTPUT_PATH = BASE_PATH + r"\bm25_subset_candidates.csv"

TOP_K = 200


print("Loading subset passages...")
passage_ids = []
passage_texts = []

with open(PASSAGES_PATH, encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    for pid, passage in reader:
        passage_ids.append(pid)
        passage_texts.append(passage)

print(f"Subset passages loaded: {len(passage_texts)}")


print("Tokenizing passages...")
tokenized_passages = [p.lower().split() for p in passage_texts]


print("Building BM25 index...")
bm25 = BM25Okapi(tokenized_passages)


print("Loading subset queries...")
queries = []

with open(QUERIES_PATH, encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    next(reader)  
    for qid, query in reader:
        queries.append((qid, query))

print(f"Subset queries loaded: {len(queries)}")


print("Running BM25 retrieval...")
results = []

for qid, query in tqdm(queries):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)

    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:TOP_K]   

    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "query_id": qid,
            "query": query,
            "passage_id": passage_ids[idx],
            "passage": passage_texts[idx],
            "bm25_score": scores[idx],
            "rank": rank
        })


df = pd.DataFrame(results)
df.to_csv(OUTPUT_PATH, index=False)

print("BM25 on subset completed.")
print(f"Saved to: {OUTPUT_PATH}")


Loading subset passages...
Subset passages loaded: 31954
Tokenizing passages...
Building BM25 index...
Loading subset queries...
Subset queries loaded: 30000
Running BM25 retrieval...


100%|██████████| 30000/30000 [1:03:22<00:00,  7.89it/s]


BM25 on subset completed.
Saved to: C:\Users\nihal\OneDrive\Desktop\NLP_Project\bm25_subset_candidates.csv
